### 1.SETUP

In [1]:
import os
import random
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import json
import joblib

warnings.filterwarnings("ignore")


In [2]:
# =========================================================
# CONFIG
# =========================================================

SEED = 42

SEQ_LEN = 18
BATCH_SIZE = 128
EPOCHS = 55
LR = 8e-4
WEIGHT_DECAY = 1e-5

HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.25
EMBED_DIM = 32

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_FILE = "/Users/dhanujiamanda/Documents/Projects/Agentic AI /Pipeline/Agentic-AI-for-Pharma-Stockout-Problem/notebooks/Forecast_Model/base_cleaned_data.csv"

EVAL_DIR = "models_gru_residual_bonus"
DEPLOY_DIR = "models_gru_residual_bonus_deploy"

TRAIN_END_YEAR = 2023
VALID_YEAR = 2024
TEST_YEAR = 2025

ABC_WEIGHT_MAP = {0: 2.5, 1: 1.2, 2: 1.0}
PROMO_WEIGHT = 1.35
SUPPLY_WEIGHT = 1.20
RECURRING_PROMO_WEIGHT = 1.25
EXPECTED_PROMO_WEIGHT = 1.20

UNDER_PENALTY = 1.75

### 2.SHARED UTILITIES

In [3]:
# =========================================================
# REPRODUCIBILITY
# =========================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# METRICS
# =========================================================
def wmape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.sum(y_true)
    if denom == 0:
        return 0.0
    return np.sum(np.abs(y_true - y_pred)) / denom * 100

def forecast_bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.sum(y_true)
    if denom == 0:
        return 0.0
    return np.sum(y_pred - y_true) / denom * 100

def underforecast_rate(y_true, y_pred):
    diff = np.asarray(y_true) - np.asarray(y_pred)
    under = np.where(diff > 0, diff, 0)
    denom = np.sum(y_true)
    if denom == 0:
        return 0.0
    return np.sum(under) / denom * 100

def evaluate_all_metrics(y_true, y_pred):
    return {
        "WMAPE": wmape(y_true, y_pred),
        "Bias": forecast_bias(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Underforecast_Rate": underforecast_rate(y_true, y_pred),
    }


# =========================================================
# HELPERS
# =========================================================
def signed_log_transform(x):
    return np.sign(x) * np.log1p(np.abs(x))

def signed_log_inverse(x):
    return np.sign(x) * np.expm1(np.abs(x))

def infer_next_year_month(year, month_number):
    month_number = int(month_number)
    year = int(year)
    if month_number == 12:
        return year + 1, 1
    return year, month_number + 1

def make_item_mapping(train_df: pd.DataFrame):
    item_codes = sorted(train_df["ItemCode"].astype(int).unique().tolist())
    return {item: i for i, item in enumerate(item_codes)}


### 3.DATA PREPARATION

In [4]:
# =========================================================
# DATA LOADING
# =========================================================
def load_data(file_path: str) -> pd.DataFrame:
    if file_path.lower().endswith(".xlsx"):
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path)

    df = df[~((df["Year"] == 2026) & (df["Month_Number"] == 2))].copy()

    df["ItemCode"] = pd.to_numeric(df["ItemCode"], errors="coerce")
    df = df.dropna(subset=["ItemCode"]).copy()
    df["ItemCode"] = df["ItemCode"].astype(int)

    numeric_cols = [
        "Secondary_Sales_Qty",
        "Primary_Sales_Qty",
        "Free_Qty",
        "Available_Primary_Inventory_Qty",
        "Distributor_Inventory_Qty",
        "Blocked_Stock_Qty",
        "Inspection_Stock_Qty",
        "Total_Primary_Inventory_Qty",
        "Observed_Demand",
        "Clean_Demand",
        "Net_Available_Stock",
        "Inventory_Pressure",
        "Stock_Cover_Months",
        "Primary_Stock_Cover",
        "Distributor_Stock_Cover",
        "Demand_to_Stock_Ratio",
        "Rolling3M_Mean",
        "Rolling6M_Mean",
        "Rolling3M_Std",
        "Promo_Uplift_Lag1",
        "Promo_Uplift_Lag2",
        "Promo_Uplift_6M",
        "Last_Bonus_Demand",
        "Avg_Bonus_Uplift",
        "Bonus_Cycle_Length",
        "Months_Since_Last_Bonus",
        "Expected_Bonus_Month",
        "Expected_Bonus_NextMonth",
        "Post_Bonus_NextMonth_Flag",
        "Recurring_Bonus_SKU",
        "Supply_Shock",
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

    if "Clean_Demand" not in df.columns:
        df["Clean_Demand"] = df["Secondary_Sales_Qty"].clip(lower=0)

    for c in ["Bonus_Flag", "Supply_Constraint_Flag", "Distributor_Buffer_Flag"]:
        if c not in df.columns:
            df[c] = 0

    # defaults for recurring bonus fields if not present
    defaults = {
        "Recurring_Bonus_SKU": 0,
        "Bonus_Cycle_Length": 0,
        "Expected_Bonus_Month": 0,
        "Expected_Bonus_NextMonth": 0,
        "Post_Bonus_NextMonth_Flag": 0,
        "Avg_Bonus_Uplift": 1.0,
        "Promo_Uplift_Lag1": 1.0,
        "Promo_Uplift_Lag2": 1.0,
        "Promo_Uplift_6M": 1.0,
        "Last_Bonus_Demand": 0.0,
        "Months_Since_Last_Bonus": 999,
        "Supply_Shock": 0,
    }
    for c, default_val in defaults.items():
        if c not in df.columns:
            df[c] = default_val
        else:
            df[c] = df[c].fillna(default_val)

    df = df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    return df

# =========================================================
# FEATURE ENGINEERING
# =========================================================
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    grp = df.groupby("ItemCode", group_keys=False)

    # seasonality
    df["Month_Sin"] = np.sin(2 * np.pi * df["Month_Number"] / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * df["Month_Number"] / 12)

    # lags
    for lag in [1, 2, 3, 6, 12]:
        if f"Lag{lag}" not in df.columns:
            df[f"Lag{lag}"] = grp["Clean_Demand"].shift(lag)

    # rolling stats
    if "Rolling3M_Mean" not in df.columns:
        df["Rolling3M_Mean"] = grp["Clean_Demand"].transform(
            lambda x: x.rolling(3, min_periods=1).mean().shift(1)
        )
    if "Rolling6M_Mean" not in df.columns:
        df["Rolling6M_Mean"] = grp["Clean_Demand"].transform(
            lambda x: x.rolling(6, min_periods=1).mean().shift(1)
        )
    if "Rolling3M_Std" not in df.columns:
        df["Rolling3M_Std"] = grp["Clean_Demand"].transform(
            lambda x: x.rolling(3, min_periods=1).std().shift(1)
        ).fillna(0)

    df["Momentum"] = df["Lag1"] - df["Lag3"]

    # stock features
    if "Net_Available_Stock" not in df.columns:
        df["Net_Available_Stock"] = (
            df["Total_Primary_Inventory_Qty"]
            - df["Blocked_Stock_Qty"]
            - df["Inspection_Stock_Qty"]
        ).clip(lower=0)

    if "Inventory_Pressure" not in df.columns:
        df["Inventory_Pressure"] = np.where(
            df["Lag1"].fillna(0) <= 0,
            0,
            df["Available_Primary_Inventory_Qty"] / (df["Lag1"] + 1)
        )

    if "Stock_Cover_Months" not in df.columns:
        df["Stock_Cover_Months"] = np.where(
            df["Rolling3M_Mean"].fillna(0) <= 0,
            0,
            df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1)
        )

    if "Primary_Stock_Cover" not in df.columns:
        df["Primary_Stock_Cover"] = np.where(
            df["Rolling3M_Mean"].fillna(0) <= 0,
            0,
            df["Net_Available_Stock"] / (df["Rolling3M_Mean"] + 1)
        )

    if "Distributor_Stock_Cover" not in df.columns:
        df["Distributor_Stock_Cover"] = np.where(
            df["Rolling3M_Mean"].fillna(0) <= 0,
            0,
            df["Distributor_Inventory_Qty"] / (df["Rolling3M_Mean"] + 1)
        )

    if "Demand_to_Stock_Ratio" not in df.columns:
        df["Demand_to_Stock_Ratio"] = np.where(
            df["Net_Available_Stock"] <= 0,
            0,
            df["Rolling3M_Mean"] / (df["Net_Available_Stock"] + 1)
        )

    # free ratio
    if "Free_Ratio" not in df.columns:
        df["Free_Ratio"] = np.where(
            df["Primary_Sales_Qty"] <= 0,
            0,
            df["Free_Qty"] / (df["Primary_Sales_Qty"] + 1)
        )

    # bonus history
    if "Bonus_Flag_Lag1" not in df.columns:
        df["Bonus_Flag_Lag1"] = grp["Bonus_Flag"].shift(1).fillna(0)
    if "Bonus_Flag_Lag2" not in df.columns:
        df["Bonus_Flag_Lag2"] = grp["Bonus_Flag"].shift(2).fillna(0)

    if "Bonus_Frequency_12M" not in df.columns:
        df["Bonus_Frequency_12M"] = grp["Bonus_Flag"].transform(
            lambda x: x.rolling(12, min_periods=1).mean().shift(1)
        ).fillna(0)

    # supply history
    if "Supply_Constraint_Lag1" not in df.columns:
        df["Supply_Constraint_Lag1"] = grp["Supply_Constraint_Flag"].shift(1).fillna(0)
    if "Supply_Constraint_Lag2" not in df.columns:
        df["Supply_Constraint_Lag2"] = grp["Supply_Constraint_Flag"].shift(2).fillna(0)

    # zero-rate
    df["Is_Zero"] = (df["Clean_Demand"] == 0).astype(int)
    if "ZeroRate_6M" not in df.columns:
        df["ZeroRate_6M"] = grp["Is_Zero"].transform(
            lambda x: x.rolling(6, min_periods=1).mean().shift(1)
        ).fillna(0)

    # cycle position feature
    df["Cycle_Position"] = np.where(
        df["Bonus_Cycle_Length"] > 0,
        df["Months_Since_Last_Bonus"] / (df["Bonus_Cycle_Length"] + 1),
        0
    )
    df["Cycle_Position"] = df["Cycle_Position"].clip(0, 5)

    # sku stats
    sku_stats = df.groupby("ItemCode").agg(
        SKU_Mean_Demand=("Clean_Demand", "mean"),
        SKU_Std_Demand=("Clean_Demand", "std"),
        SKU_ZeroRate=("Is_Zero", "mean"),
        SKU_BonusRate=("Bonus_Flag", "mean"),
        SKU_SupplyConstraintRate=("Supply_Constraint_Flag", "mean"),
    ).reset_index()

    sku_stats["SKU_CV"] = np.where(
        sku_stats["SKU_Mean_Demand"] <= 0,
        0,
        sku_stats["SKU_Std_Demand"].fillna(0) / (sku_stats["SKU_Mean_Demand"] + 1)
    )

    df = df.drop(
        columns=["SKU_Mean_Demand", "SKU_Std_Demand", "SKU_ZeroRate", "SKU_BonusRate", "SKU_SupplyConstraintRate", "SKU_CV"],
        errors="ignore"
    )

    df = df.merge(
        sku_stats[
            ["ItemCode", "SKU_Mean_Demand", "SKU_ZeroRate", "SKU_BonusRate", "SKU_SupplyConstraintRate", "SKU_CV"]
        ],
        on="ItemCode",
        how="left",
    )

    # ABC if not present
    if "ABC_Class" not in df.columns:
        sku_total = df.groupby("ItemCode")["Clean_Demand"].sum().sort_values(ascending=False)
        cum_pct = sku_total.cumsum() / sku_total.sum()
        abc_series = pd.cut(cum_pct, bins=[0, 0.7, 0.9, 1.0], labels=[0, 1, 2])
        abc_map = abc_series.to_dict()
        df["ABC_Class"] = df["ItemCode"].map(abc_map).fillna(2).astype(int)
    else:
        df["ABC_Class"] = df["ABC_Class"].fillna(2).astype(int)

    # target
    grp = df.groupby("ItemCode", group_keys=False)
    df["Target"] = grp["Clean_Demand"].shift(-1)

    # residual target over rolling baseline
    # use current row baseline to predict next month deviation
    df["Residual_Baseline"] = df["Rolling3M_Mean"].fillna(df["Lag1"]).fillna(0)

    df["Residual_Target"] = df["Target"] - df["Residual_Baseline"]

    # signed log transform for residuals
    df["Residual_Target_Log"] = np.sign(df["Residual_Target"]) * np.log1p(np.abs(df["Residual_Target"]))

    df = df.replace([np.inf, -np.inf], np.nan)
    return df


In [5]:
# =========================================================
# FEATURE SETS
# =========================================================
SEQ_FEATURES = [
    "Clean_Demand",
    "Secondary_Sales_Qty",
    "Primary_Sales_Qty",
    "Free_Qty",
    "Free_Ratio",
    "Bonus_Flag",
    "Bonus_Flag_Lag1",
    "Bonus_Flag_Lag2",
    "Bonus_Frequency_12M",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Months_Since_Last_Bonus",
    "Cycle_Position",
    "Expected_Bonus_Month",
    "Expected_Bonus_NextMonth",
    "Post_Bonus_NextMonth_Flag",
    "Avg_Bonus_Uplift",
    "Promo_Uplift_Lag1",
    "Promo_Uplift_Lag2",
    "Promo_Uplift_6M",
    "Last_Bonus_Demand",
    "Supply_Constraint_Flag",
    "Supply_Constraint_Lag1",
    "Supply_Constraint_Lag2",
    "Supply_Shock",
    "Distributor_Buffer_Flag",
    "Available_Primary_Inventory_Qty",
    "Distributor_Inventory_Qty",
    "Net_Available_Stock",
    "Inventory_Pressure",
    "Stock_Cover_Months",
    "Primary_Stock_Cover",
    "Distributor_Stock_Cover",
    "Demand_to_Stock_Ratio",
    "Lag1",
    "Lag2",
    "Lag3",
    "Lag6",
    "Lag12",
    "Rolling3M_Mean",
    "Rolling6M_Mean",
    "Rolling3M_Std",
    "Momentum",
    "Month_Sin",
    "Month_Cos",
    "ZeroRate_6M",
]

STATIC_FEATURES = [
    "ABC_Class",
    "SKU_Mean_Demand",
    "SKU_ZeroRate",
    "SKU_BonusRate",
    "SKU_SupplyConstraintRate",
    "SKU_CV",
    "Recurring_Bonus_SKU",
    "Bonus_Cycle_Length",
    "Avg_Bonus_Uplift",
]

TARGET_COL = "Target"
BASELINE_COL = "Residual_Baseline"
RESIDUAL_COL = "Residual_Target"
RESIDUAL_LOG_COL = "Residual_Target_Log"


### 4.MODEL COMPONENTS

In [6]:
# =========================================================
# DATASET
# =========================================================
class PharmaSequenceDataset(Dataset):
    def __init__(self, X_seq, X_static, X_item, y_res_log, sample_w):
        self.X_seq = X_seq
        self.X_static = X_static
        self.X_item = X_item
        self.y_res_log = y_res_log
        self.sample_w = sample_w

    def __len__(self):
        return len(self.y_res_log)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X_seq[idx], dtype=torch.float32),
            torch.tensor(self.X_static[idx], dtype=torch.float32),
            torch.tensor(self.X_item[idx], dtype=torch.long),
            torch.tensor(self.y_res_log[idx], dtype=torch.float32),
            torch.tensor(self.sample_w[idx], dtype=torch.float32),
        )

# =========================================================
# MODEL
# =========================================================
class GlobalGRUResidualForecaster(nn.Module):
    def __init__(
        self,
        num_items: int,
        seq_input_dim: int,
        static_input_dim: int,
        embed_dim: int = 32,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.25,
    ):
        super().__init__()

        self.item_embedding = nn.Embedding(num_items, embed_dim)

        self.gru = nn.GRU(
            input_size=seq_input_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.seq_fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.static_fc = nn.Sequential(
            nn.Linear(static_input_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Sequential(
            nn.Linear(64 + 32 + embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x_seq, x_static, x_item):
        out, _ = self.gru(x_seq)
        last_hidden = out[:, -1, :]

        seq_repr = self.seq_fc(last_hidden)
        static_repr = self.static_fc(x_static)
        item_repr = self.item_embedding(x_item)

        x = torch.cat([seq_repr, static_repr, item_repr], dim=1)
        pred_res_log = self.head(x).squeeze(1)
        return pred_res_log

# =========================================================
# LOSS
# =========================================================
class WeightedAsymmetricMAELoss(nn.Module):
    def __init__(self, under_penalty: float = 1.75):
        super().__init__()
        self.under_penalty = under_penalty

    def forward(self, preds, targets, sample_weights):
        err = preds - targets
        abs_err = torch.abs(err)
        penalty = torch.where(err < 0, self.under_penalty, 1.0)
        loss = abs_err * penalty * sample_weights
        return loss.mean()


### 5.Eval PIPELINE

In [7]:
# =========================================================
# SPLITS
# =========================================================
def split_train_valid_test(df: pd.DataFrame):
    train_df = df[df["Year"] <= TRAIN_END_YEAR].copy()
    valid_df = df[df["Year"] == VALID_YEAR].copy()
    test_df = df[df["Year"] == TEST_YEAR].copy()

    if train_df.empty or valid_df.empty or test_df.empty:
        raise ValueError("One of train/valid/test splits is empty. Check years.")

    return train_df, valid_df, test_df

# =========================================================
# SCALERS
# =========================================================
@dataclass
class ScalerBundle:
    seq_scaler: StandardScaler
    static_scaler: StandardScaler

def fit_eval_scalers(train_df: pd.DataFrame) -> ScalerBundle:
    seq_scaler = StandardScaler()
    static_scaler = StandardScaler()

    seq_scaler.fit(train_df[SEQ_FEATURES].fillna(0))
    static_scaler.fit(train_df[STATIC_FEATURES].fillna(0))

    return ScalerBundle(seq_scaler=seq_scaler, static_scaler=static_scaler)

# =========================================================
# SEQUENCE BUILD
# =========================================================
def build_sequences_full(df: pd.DataFrame, item_to_idx: dict, scalers: ScalerBundle, seq_len: int = 18):
    X_seq, X_static, X_item = [], [], []
    y_res_log, sample_w = [], []
    meta = []

    for item, g in df.groupby("ItemCode"):
        if item not in item_to_idx:
            continue

        g = g.sort_values(["Year", "Month_Number"]).copy().reset_index(drop=True)
        usable_idx = g.index[g[RESIDUAL_LOG_COL].notna()].tolist()

        for idx in usable_idx:
            start = idx - seq_len + 1
            if start < 0:
                continue

            seq_slice = g.iloc[start:idx + 1].copy()
            if len(seq_slice) != seq_len:
                continue

            seq_vals = seq_slice[SEQ_FEATURES].fillna(0).values
            seq_vals = scalers.seq_scaler.transform(seq_vals)

            static_vals = seq_slice.iloc[-1][STATIC_FEATURES].fillna(0).values.reshape(1, -1)
            static_vals = scalers.static_scaler.transform(static_vals)[0]

            y_val = float(g.iloc[idx][RESIDUAL_LOG_COL])

            abc_class = int(seq_slice.iloc[-1]["ABC_Class"])
            bonus_flag = int(seq_slice.iloc[-1]["Bonus_Flag"])
            supply_flag = int(seq_slice.iloc[-1]["Supply_Constraint_Flag"])
            recurring_flag = int(seq_slice.iloc[-1]["Recurring_Bonus_SKU"])
            expected_bonus_flag = int(seq_slice.iloc[-1]["Expected_Bonus_NextMonth"])

            w = ABC_WEIGHT_MAP.get(abc_class, 1.0)
            if bonus_flag == 1:
                w *= PROMO_WEIGHT
            if supply_flag == 1:
                w *= SUPPLY_WEIGHT
            if recurring_flag == 1:
                w *= RECURRING_PROMO_WEIGHT
            if expected_bonus_flag == 1:
                w *= EXPECTED_PROMO_WEIGHT

            X_seq.append(seq_vals.astype(np.float32))
            X_static.append(static_vals.astype(np.float32))
            X_item.append(item_to_idx[item])
            y_res_log.append(np.float32(y_val))
            sample_w.append(np.float32(w))

            meta.append({
                "ItemCode": int(item),
                "Target_Year": int(g.iloc[idx]["Year"]),
                "Target_Month_Number": int(g.iloc[idx]["Month_Number"]),
                "ABC_Class": abc_class,
                "Bonus_Flag": bonus_flag,
                "Supply_Constraint_Flag": supply_flag,
                "Recurring_Bonus_SKU": recurring_flag,
                "Expected_Bonus_NextMonth": expected_bonus_flag,
                "Target_Actual": float(g.iloc[idx][TARGET_COL]),
                "Residual_Baseline": float(g.iloc[idx][BASELINE_COL]),
                "Residual_Target": float(g.iloc[idx][RESIDUAL_COL]),
            })

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(X_item, dtype=np.int64),
        np.array(y_res_log, dtype=np.float32),
        np.array(sample_w, dtype=np.float32),
        pd.DataFrame(meta),
    )

def split_sequence_arrays(X_seq, X_static, X_item, y, w, meta_df):
    train_mask = meta_df["Target_Year"] <= TRAIN_END_YEAR
    valid_mask = meta_df["Target_Year"] == VALID_YEAR
    test_mask  = meta_df["Target_Year"] == TEST_YEAR

    def take(mask):
        idx = np.where(mask.values)[0]
        return (
            X_seq[idx],
            X_static[idx],
            X_item[idx],
            y[idx],
            w[idx],
            meta_df.iloc[idx].reset_index(drop=True)
        )

    return take(train_mask), take(valid_mask), take(test_mask)

# =========================================================
# TRAIN / EVAL HELPERS
# =========================================================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(DEVICE)
        x_static = x_static.to(DEVICE)
        x_item = x_item.to(DEVICE)
        y_res_log = y_res_log.to(DEVICE)
        sample_w = sample_w.to(DEVICE)

        optimizer.zero_grad()
        preds = model(x_seq, x_static, x_item)
        loss = criterion(preds, y_res_log, sample_w)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * len(y_res_log)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def predict_residual_log(model, loader):
    model.eval()

    preds_all, y_all = [], []

    for x_seq, x_static, x_item, y_res_log, sample_w in loader:
        x_seq = x_seq.to(DEVICE)
        x_static = x_static.to(DEVICE)
        x_item = x_item.to(DEVICE)

        preds = model(x_seq, x_static, x_item).cpu().numpy()
        preds_all.append(preds)
        y_all.append(y_res_log.numpy())

    preds_all = np.concatenate(preds_all)
    y_all = np.concatenate(y_all)
    return preds_all, y_all

def evaluate_on_loader(model, loader, meta_df):
    pred_res_log, true_res_log = predict_residual_log(model, loader)

    pred_residual = signed_log_inverse(pred_res_log)
    true_residual = signed_log_inverse(true_res_log)

    out = meta_df.copy()
    out["Pred_Residual_Log"] = pred_res_log
    out["Pred_Residual"] = pred_residual
    out["True_Residual"] = true_residual

    out["Pred"] = out["Residual_Baseline"] + out["Pred_Residual"]
    out["Pred"] = out["Pred"].clip(lower=0)

    out["Error"] = out["Target_Actual"] - out["Pred"]
    out["Abs_Error"] = np.abs(out["Error"])

    metrics = evaluate_all_metrics(out["Target_Actual"].values, out["Pred"].values)
    return metrics, out

def fit_eval_model(train_dataset, valid_dataset, valid_meta, num_items, seq_input_dim, static_input_dim):
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = GlobalGRUResidualForecaster(
        num_items=num_items,
        seq_input_dim=seq_input_dim,
        static_input_dim=static_input_dim,
        embed_dim=EMBED_DIM,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=UNDER_PENALTY)

    best_valid_wmape = float("inf")
    best_state = None
    patience = 10
    wait = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        valid_metrics, _ = evaluate_on_loader(model, valid_loader, valid_meta)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Valid WMAPE: {valid_metrics['WMAPE']:.4f} | "
            f"Valid Bias: {valid_metrics['Bias']:.4f}"
        )

        if valid_metrics["WMAPE"] < best_valid_wmape:
            best_valid_wmape = valid_metrics["WMAPE"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

# =========================================================
# MAIN PIPELINE
# =========================================================
def run_eval_pipeline():
    print("Loading data...")
    df = load_data(DATA_FILE)

    print("Building features...")
    df = build_features(df)

    df = df.dropna(subset=["Year", "Month_Number", "ItemCode"]).copy()

    train_df, valid_df, test_df = split_train_valid_test(df)

    print("Fitting scalers on train only...")
    scalers = fit_eval_scalers(train_df)

    print("Building item mapping...")
    item_to_idx = make_item_mapping(train_df)

    df = df[df["ItemCode"].isin(item_to_idx.keys())].copy()
    
    print("Building full sequences...")
    X_seq_all, X_static_all, X_item_all, y_all, w_all, meta_all = build_sequences_full(df, item_to_idx, scalers, seq_len=SEQ_LEN)

    (train_pack, valid_pack, test_pack) = split_sequence_arrays(X_seq_all, X_static_all, X_item_all, y_all, w_all, meta_all)

    X_seq_train, X_static_train, X_item_train, y_train, w_train, meta_train = train_pack
    X_seq_valid, X_static_valid, X_item_valid, y_valid, w_valid, meta_valid = valid_pack
    X_seq_test, X_static_test, X_item_test, y_test, w_test, meta_test = test_pack


    if len(y_train) == 0 or len(y_valid) == 0 or len(y_test) == 0:
        raise ValueError("No sequences were created. Check seq length or year coverage.")

    print("Train sequences:", len(y_train))
    print("Valid sequences:", len(y_valid))
    print("Test sequences:", len(y_test))

    train_dataset = PharmaSequenceDataset(X_seq_train, X_static_train, X_item_train, y_train, w_train)
    valid_dataset = PharmaSequenceDataset(X_seq_valid, X_static_valid, X_item_valid, y_valid, w_valid)
    test_dataset = PharmaSequenceDataset(X_seq_test, X_static_test, X_item_test, y_test, w_test)

    print("Training residual GRU model...")
    model = fit_eval_model(
        train_dataset=train_dataset,
        valid_dataset=valid_dataset,
        valid_meta=meta_valid,
        num_items=len(item_to_idx),
        seq_input_dim=X_seq_train.shape[2],
        static_input_dim=X_static_train.shape[1],
    )

    print("Evaluating on validation and test...")
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    valid_metrics, valid_result_df = evaluate_on_loader(model, valid_loader, meta_valid)
    test_metrics, test_result_df = evaluate_on_loader(model, test_loader, meta_test)

    print("\n===== VALID METRICS (2024) =====")
    for k, v in valid_metrics.items():
        print(f"{k}: {v:.4f}")

    print("\n===== TEST METRICS (2025) =====")
    for k, v in test_metrics.items():
        print(f"{k}: {v:.4f}")

    print("\n===== 2025 WMAPE by ABC =====")
    print(
        test_result_df.groupby("ABC_Class").apply(
            lambda x: wmape(x["Target_Actual"].values, x["Pred"].values)
        )
    )

    os.makedirs(EVAL_DIR, exist_ok=True)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "item_to_idx": item_to_idx,
            "seq_features": SEQ_FEATURES,
            "static_features": STATIC_FEATURES,
            "seq_len": SEQ_LEN,
            "embed_dim": EMBED_DIM,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
        },
        os.path.join(EVAL_DIR, "global_gru_residual_bonus.pt")
    )

    valid_result_df.to_csv(os.path.join(EVAL_DIR, "valid_predictions_2024.csv"), index=False)
    test_result_df.to_csv(os.path.join(EVAL_DIR, "test_predictions_2025.csv"), index=False)

    print("\nSaved:")
    print(f" - {os.path.join(EVAL_DIR, 'global_gru_residual_bonus.pt')}")
    print(f" - {os.path.join(EVAL_DIR, 'valid_predictions_2024.csv')}")
    print(f" - {os.path.join(EVAL_DIR, 'test_predictions_2025.csv')}")

    return model, valid_result_df, test_result_df, valid_metrics, test_metrics


### 6.RUN EVAL

In [8]:
# =========================================================
# GRU: LONGER SEQUENCE + RECURRING BONUS FEATURES + RESIDUAL TARGET
# =========================================================

seed_everything(SEED)

# =========================================================
# RUN
# =========================================================
model, valid_df_out, test_df_out, valid_metrics, test_metrics = run_eval_pipeline()

Loading data...
Building features...
Fitting scalers on train only...
Building item mapping...
Building full sequences...
Train sequences: 8916
Valid sequences: 6434
Test sequences: 6568
Training residual GRU model...
Epoch 01 | Train Loss: 13.48493 | Valid WMAPE: 27.9468 | Valid Bias: 9.6588
Epoch 02 | Train Loss: 12.30070 | Valid WMAPE: 25.8531 | Valid Bias: 3.0509
Epoch 03 | Train Loss: 12.24158 | Valid WMAPE: 25.7068 | Valid Bias: 2.7181
Epoch 04 | Train Loss: 12.12224 | Valid WMAPE: 25.3170 | Valid Bias: 1.1900
Epoch 05 | Train Loss: 11.89813 | Valid WMAPE: 25.3825 | Valid Bias: 4.3952
Epoch 06 | Train Loss: 11.76506 | Valid WMAPE: 26.0951 | Valid Bias: 6.8330
Epoch 07 | Train Loss: 11.61743 | Valid WMAPE: 24.7680 | Valid Bias: 1.9570
Epoch 08 | Train Loss: 11.40664 | Valid WMAPE: 25.5130 | Valid Bias: 5.7169
Epoch 09 | Train Loss: 10.97440 | Valid WMAPE: 25.0105 | Valid Bias: 2.7171
Epoch 10 | Train Loss: 10.86915 | Valid WMAPE: 25.2764 | Valid Bias: 4.2905
Epoch 11 | Train Loss:

In [9]:
# worst SKUs
test_df_out.sort_values("Abs_Error", ascending=False).head(20)

# recurring bonus SKUs
test_df_out[test_df_out["Recurring_Bonus_SKU"] == 1]

# ABC breakdown
test_df_out.groupby("ABC_Class").apply(
    lambda x: wmape(x["Target_Actual"], x["Pred"])
)

ABC_Class
0    23.385469
1    25.346706
2    27.559239
dtype: float64

### 7.DeployMENT PIPELINE

In [10]:
# =========================================================
# TRAIN DATA BUILDER
# =========================================================
def fit_deploy_scalers(full_train_df: pd.DataFrame):
    seq_scaler = StandardScaler()
    static_scaler = StandardScaler()

    seq_scaler.fit(full_train_df[SEQ_FEATURES].fillna(0))
    static_scaler.fit(full_train_df[STATIC_FEATURES].fillna(0))

    return seq_scaler, static_scaler

def build_deploy_sequences(df: pd.DataFrame, item_to_idx: dict, seq_scaler, static_scaler, seq_len: int):
    X_seq, X_static, X_item, y, w = [], [], [], [], []

    for item, g in df.groupby("ItemCode"):
        if item not in item_to_idx:
            continue

        g = g.sort_values(["Year", "Month_Number"]).copy().reset_index(drop=True)
        usable_idx = g.index[g["Residual_Target_Log"].notna()].tolist()

        for idx in usable_idx:
            start = idx - seq_len + 1
            if start < 0:
                continue

            seq_slice = g.iloc[start:idx + 1].copy()
            if len(seq_slice) != seq_len:
                continue

            seq_vals = seq_scaler.transform(seq_slice[SEQ_FEATURES].fillna(0).values)
            static_vals = static_scaler.transform(
                seq_slice.iloc[-1][STATIC_FEATURES].fillna(0).values.reshape(1, -1)
            )[0]

            target_val = float(g.iloc[idx]["Residual_Target_Log"])

            abc_class = int(seq_slice.iloc[-1]["ABC_Class"])
            bonus_flag = int(seq_slice.iloc[-1]["Bonus_Flag"])
            supply_flag = int(seq_slice.iloc[-1]["Supply_Constraint_Flag"])
            recurring_flag = int(seq_slice.iloc[-1]["Recurring_Bonus_SKU"])
            expected_bonus_flag = int(seq_slice.iloc[-1]["Expected_Bonus_NextMonth"])

            sample_weight = ABC_WEIGHT_MAP.get(abc_class, 1.0)
            if bonus_flag == 1:
                sample_weight *= PROMO_WEIGHT
            if supply_flag == 1:
                sample_weight *= SUPPLY_WEIGHT
            if recurring_flag == 1:
                sample_weight *= RECURRING_PROMO_WEIGHT
            if expected_bonus_flag == 1:
                sample_weight *= EXPECTED_PROMO_WEIGHT

            X_seq.append(seq_vals.astype(np.float32))
            X_static.append(static_vals.astype(np.float32))
            X_item.append(item_to_idx[item])
            y.append(np.float32(target_val))
            w.append(np.float32(sample_weight))

    return (
        np.array(X_seq, dtype=np.float32),
        np.array(X_static, dtype=np.float32),
        np.array(X_item, dtype=np.int64),
        np.array(y, dtype=np.float32),
        np.array(w, dtype=np.float32),
    )

# =========================================================
# TRAIN DEPLOYMENT MODEL
# =========================================================
def train_deployment_model():
    os.makedirs(DEPLOY_DIR, exist_ok=True)

    print("Loading full complete data...")
    df = load_data(DATA_FILE)

    print("Building deployment features...")
    df = build_features(df)

    print("Creating item mapping...")
    item_to_idx = make_item_mapping(df)

    print("Fitting scalers...")
    seq_scaler, static_scaler = fit_deploy_scalers(df)

    print("Building train sequences...")
    X_seq, X_static, X_item, y, w = build_deploy_sequences(
        df, item_to_idx, seq_scaler, static_scaler, seq_len=SEQ_LEN
    )

    print("Train sequences:", len(y))
    if len(y) == 0:
        raise ValueError("No deployment sequences created.")

    dataset = PharmaSequenceDataset(X_seq, X_static, X_item, y, w)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = GlobalGRUResidualForecaster(
        num_items=len(item_to_idx),
        seq_input_dim=len(SEQ_FEATURES),
        static_input_dim=len(STATIC_FEATURES),
        embed_dim=EMBED_DIM,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = WeightedAsymmetricMAELoss(under_penalty=UNDER_PENALTY)

    print("Training deployment model...")
    best_loss = float("inf")
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for x_seq, x_static, x_item, y_batch, w_batch in loader:
            x_seq = x_seq.to(DEVICE)
            x_static = x_static.to(DEVICE)
            x_item = x_item.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            w_batch = w_batch.to(DEVICE)

            optimizer.zero_grad()
            preds = model(x_seq, x_static, x_item)
            loss = criterion(preds, y_batch, w_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item() * len(y_batch)

        epoch_loss = total_loss / len(loader.dataset)
        print(f"Epoch {epoch:02d} | Train Loss: {epoch_loss:.5f}")

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    print("Saving deployment artifacts...")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "seq_len": SEQ_LEN,
            "embed_dim": EMBED_DIM,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "seq_features": SEQ_FEATURES,
            "static_features": STATIC_FEATURES,
            "item_to_idx": item_to_idx,
        },
        os.path.join(DEPLOY_DIR, "deploy_model.pt")
    )

    joblib.dump(seq_scaler, os.path.join(DEPLOY_DIR, "seq_scaler.pkl"))
    joblib.dump(static_scaler, os.path.join(DEPLOY_DIR, "static_scaler.pkl"))

    with open(os.path.join(DEPLOY_DIR, "deploy_config.json"), "w") as f:
        json.dump({
            "seq_len": SEQ_LEN,
            "embed_dim": EMBED_DIM,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "dropout": DROPOUT,
            "seq_features": SEQ_FEATURES,
            "static_features": STATIC_FEATURES,
        }, f, indent=2)

    print("Saved:")
    print(f" - {DEPLOY_DIR}/deploy_model.pt")
    print(f" - {DEPLOY_DIR}/seq_scaler.pkl")
    print(f" - {DEPLOY_DIR}/static_scaler.pkl")
    print(f" - {DEPLOY_DIR}/deploy_config.json")

# =========================================================
# LOAD DEPLOYMENT MODEL
# =========================================================
def load_deployment_artifacts():
    artifact = torch.load(os.path.join(DEPLOY_DIR, "deploy_model.pt"), map_location=DEVICE)
    seq_scaler = joblib.load(os.path.join(DEPLOY_DIR, "seq_scaler.pkl"))
    static_scaler = joblib.load(os.path.join(DEPLOY_DIR, "static_scaler.pkl"))

    model = GlobalGRUResidualForecaster(
        num_items=len(artifact["item_to_idx"]),
        seq_input_dim=len(artifact["seq_features"]),
        static_input_dim=len(artifact["static_features"]),
        embed_dim=artifact["embed_dim"],
        hidden_size=artifact["hidden_size"],
        num_layers=artifact["num_layers"],
        dropout=artifact["dropout"],
    ).to(DEVICE)

    model.load_state_dict(artifact["model_state_dict"])
    model.eval()

    return model, artifact, seq_scaler, static_scaler

# =========================================================
# FUTURE BONUS INFERENCE
# =========================================================
def infer_expected_bonus_next_month(g):
    g = g.sort_values(["Year", "Month_Number"]).copy()
    if g.empty:
        return 0

    recurring = int(g["Recurring_Bonus_SKU"].iloc[-1]) if "Recurring_Bonus_SKU" in g.columns else 0
    cycle_len = float(g["Bonus_Cycle_Length"].iloc[-1]) if "Bonus_Cycle_Length" in g.columns else 0
    months_since = float(g["Months_Since_Last_Bonus"].iloc[-1]) if "Months_Since_Last_Bonus" in g.columns else 999

    if recurring == 1 and cycle_len > 0:
        if abs((months_since + 1) - cycle_len) <= 1:
            return 1
    return 0

# =========================================================
# BUILD FUTURE ROW
# =========================================================
def build_future_frame_for_sku(df, sku_code, next_month_bonus=None):
    sku_code = int(sku_code)
    g = df[df["ItemCode"] == sku_code].copy().sort_values(["Year", "Month_Number"]).reset_index(drop=True)
    if g.empty:
        return None

    last_row = g.iloc[-1].copy()
    next_year, next_month = infer_next_year_month(last_row["Year"], last_row["Month_Number"])

    if next_month_bonus is None:
        next_month_bonus = infer_expected_bonus_next_month(g)

    new_row = last_row.copy()
    new_row["Year"] = next_year
    new_row["Month_Number"] = next_month
    new_row["Month"] = f"{next_year}-{str(next_month).zfill(2)}"
    new_row["Bonus_Flag"] = int(next_month_bonus)

    # demand-like columns unknown in future row; keep as 0 or carry context-safe values
    for c in ["Secondary_Sales_Qty", "Primary_Sales_Qty", "Free_Qty", "Observed_Demand", "Clean_Demand"]:
        if c in new_row.index:
            new_row[c] = 0

    work_df = pd.concat([df.copy(), pd.DataFrame([new_row])], ignore_index=True)
    work_df = work_df.sort_values(["ItemCode", "Year", "Month_Number"]).reset_index(drop=True)
    work_df = build_features(work_df)

    return work_df, next_year, next_month

# =========================================================
# SINGLE SKU NEXT-MONTH FORECAST
# =========================================================
def forecast_next_month_for_sku(sku_code, raw_df=None, next_month_bonus=None):
    if raw_df is None:
        raw_df = load_data(DATA_FILE)

    model, artifact, seq_scaler, static_scaler = load_deployment_artifacts()

    work = build_future_frame_for_sku(raw_df, sku_code, next_month_bonus=next_month_bonus)
    if work is None:
        return None

    work_df, next_year, next_month = work
    g = work_df[work_df["ItemCode"] == int(sku_code)].copy().sort_values(["Year", "Month_Number"]).reset_index(drop=True)

    target_idx = g[(g["Year"] == next_year) & (g["Month_Number"] == next_month)].index
    if len(target_idx) == 0:
        return None
    idx = target_idx[0]

    if idx < artifact["seq_len"] - 1:
        return None

    seq_slice = g.iloc[idx - artifact["seq_len"] + 1: idx + 1].copy()
    if len(seq_slice) != artifact["seq_len"]:
        return None

    seq_vals = seq_scaler.transform(seq_slice[artifact["seq_features"]].fillna(0).values)
    static_vals = static_scaler.transform(
        seq_slice.iloc[-1][artifact["static_features"]].fillna(0).values.reshape(1, -1)
    )[0]

    item_idx = artifact["item_to_idx"].get(int(sku_code))
    if item_idx is None:
        return None

    x_seq = torch.tensor(seq_vals[np.newaxis, :, :], dtype=torch.float32).to(DEVICE)
    x_static = torch.tensor(static_vals[np.newaxis, :], dtype=torch.float32).to(DEVICE)
    x_item = torch.tensor([item_idx], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        pred_res_log = model(x_seq, x_static, x_item).cpu().numpy()[0]

    pred_residual = signed_log_inverse(pred_res_log)
    target_row = g.iloc[idx]
    baseline = float(target_row["Residual_Baseline"])
    pred = max(baseline + pred_residual, 0)

    return {
        "ItemCode": int(sku_code),
        "Forecast_Year": int(next_year),
        "Forecast_Month": int(next_month),
        "Expected_Bonus": int(target_row["Bonus_Flag"]),
        "Residual_Baseline": baseline,
        "Predicted_Residual": float(pred_residual),
        "GRU_Residual_Prediction": float(pred),
        "Recurring_Bonus_SKU": int(target_row["Recurring_Bonus_SKU"]),
        "ABC_Class": int(target_row["ABC_Class"]),
    }

# =========================================================
# BULK FORECAST
# =========================================================
def forecast_next_month_bulk(raw_df=None):
    if raw_df is None:
        raw_df = load_data(DATA_FILE)

    sku_list = sorted(raw_df["ItemCode"].unique().tolist())
    rows = []
    failed = []

    for sku in sku_list:
        try:
            out = forecast_next_month_for_sku(sku, raw_df=raw_df)
            if out is not None:
                rows.append(out)
            else:
                failed.append({"ItemCode": sku, "Status": "Forecast failed"})
        except Exception as e:
            failed.append({"ItemCode": sku, "Status": str(e)})

    forecast_df = pd.DataFrame(rows).sort_values(
        ["Forecast_Year", "Forecast_Month", "ItemCode"]
    ).reset_index(drop=True) if rows else pd.DataFrame()

    failed_df = pd.DataFrame(failed)
    return forecast_df, failed_df


### 8.RUN DEPLOYMENT

In [11]:
# =========================================================
# DEPLOYMENT TRAINING + INFERENCE
# RESIDUAL GRU BONUS MODEL
# =========================================================

seed_everything(SEED)

# =========================================================
# RUN EXAMPLE
# =========================================================
train_deployment_model()
single_pred = forecast_next_month_for_sku(604664)
bulk_forecast_df, failed_df = forecast_next_month_bulk()

Loading full complete data...
Building deployment features...
Creating item mapping...
Fitting scalers...
Building train sequences...
Train sequences: 21961
Training deployment model...
Epoch 01 | Train Loss: 12.57428
Epoch 02 | Train Loss: 11.89881
Epoch 03 | Train Loss: 11.65624
Epoch 04 | Train Loss: 11.33062
Epoch 05 | Train Loss: 10.99811
Epoch 06 | Train Loss: 10.82110
Epoch 07 | Train Loss: 10.61054
Epoch 08 | Train Loss: 10.47833
Epoch 09 | Train Loss: 10.29336
Epoch 10 | Train Loss: 10.17154
Epoch 11 | Train Loss: 10.12426
Epoch 12 | Train Loss: 9.99750
Epoch 13 | Train Loss: 9.92369
Epoch 14 | Train Loss: 9.84194
Epoch 15 | Train Loss: 9.80461
Epoch 16 | Train Loss: 9.78533
Epoch 17 | Train Loss: 9.68618
Epoch 18 | Train Loss: 9.68840
Epoch 19 | Train Loss: 9.57108
Epoch 20 | Train Loss: 9.49947
Epoch 21 | Train Loss: 9.40926
Epoch 22 | Train Loss: 9.38739
Epoch 23 | Train Loss: 9.27153
Epoch 24 | Train Loss: 9.23970
Epoch 25 | Train Loss: 9.22912
Epoch 26 | Train Loss: 9.084

### INFERENCE

In [12]:
df = load_data(DATA_FILE)

print(df.sort_values(["Year", "Month_Number"]).tail(5)[["Year","Month_Number"]])

       Year  Month_Number
33494  2026             1
33504  2026             1
33514  2026             1
33521  2026             1
33528  2026             1


In [13]:
PHARMA_SKUS = [
    600308, 600310, 600311, 600315, 600319, 600457, 600458, 600459, 600460, 600461,
    600462, 600478, 600481, 600542, 600547, 600586, 600589, 600600, 600601, 600602,
    600603, 600604, 600607, 600612, 600613, 600614, 600615, 600616, 600617, 600618,
    600620, 600621, 600622, 600631, 600654, 600660, 600667, 600694, 600695, 600698,
    600700, 600701, 600702, 600706, 600707, 600716, 600719, 600720, 600721, 600724,
    600726, 600822, 600823, 600828, 600830, 600831, 600837, 600914, 600919, 600921,
    600929, 600931, 600932, 601338, 601339, 601348, 601349, 601350, 601351, 601352,
    601506, 602434, 602435, 602468, 602512, 602561, 602562, 602610, 602647, 602661,
    602662, 602663, 602673, 602674, 602685, 602854, 602868, 602881, 602882, 602948,
    602949, 603644, 603652, 603692, 603703, 603706, 603755, 603773, 603774, 603775,
    603981, 603990, 603991, 604003, 604004, 604023, 604035, 604058, 604066, 604067,
    604096, 604098, 604099, 604100, 604109, 604145, 604146, 604149, 604152, 604155,
    604156, 604170, 604177, 605133, 605141, 605144, 605156, 605157, 605158, 605160,
    605162, 605163, 605164, 605165, 605168, 605173, 605174, 605181, 605182, 605183,
    605187, 605189, 605190, 605206, 605207, 605208, 605209, 605210, 605223, 605224,
    605225, 605349, 605355, 605356, 605357, 605358, 605374, 605375, 605423, 605432,
    605507, 605508, 605510, 605511, 605534, 605535, 605542, 605543, 605544, 605545,
    606273, 606307, 606353, 606363, 606365, 606395, 606397, 606398, 606402, 606413,
    606414, 606415, 606427, 606428, 606430, 606431, 606558, 606572, 606573, 606598,
    606625, 606636, 606637, 606638, 606646, 606650, 606651, 606653, 606654, 606657,
    606659, 606660, 606662, 606666, 606669, 606675, 606676, 606683, 606713, 601112,
    601115, 601131, 601133, 601134, 601137, 601140, 601141, 601142, 601143, 601330,
    601336, 601337, 601865, 601868, 601869, 601937, 601938, 601973, 601974, 602180,
    602324, 602386, 602387, 602388, 602413, 602416, 602417, 603058, 603106, 603108,
    603109, 603126, 603168, 603169, 603196, 603203, 603205, 603206, 603221, 603318,
    603321, 603322, 603323, 603324, 603378, 603434, 603493, 603551, 603581, 603582,
    603612, 603620, 603640, 604401, 604464, 604465, 604468, 604470, 604585, 604610,
    604621, 604623, 604624, 604664, 604666, 604737, 604738, 604774, 604793, 604808,
    604810, 604924, 604925, 604968, 604979, 604986, 605556, 605565, 605583, 605584,
    605613, 605622, 605657, 605685, 605703, 605927, 605935, 605936, 605937, 605973,
    606043, 606044, 606063, 606066, 606068, 606075, 606118, 606120, 606122, 606153,
    606155, 606163, 606164, 606166, 606168, 606169, 606174, 606177, 606179, 606180,
    606181, 606182, 606194, 606756, 606757, 606758, 606783, 606785, 606810, 606838,
    606840, 606843, 606841, 606844, 606897, 606899, 606900, 606910, 606927, 606991,
    606994, 606995, 606997, 606999, 607000, 607002, 607008, 607011, 607014, 607015,
    607016, 607017, 607024, 607027, 607028, 607030, 607033, 607034, 607043, 606950,
    607049, 607053, 607055, 607056, 607057, 607060, 607063, 607068, 607069, 607070,
    607071, 607072, 607074, 607076, 607079, 607087, 607089, 607090, 607091, 607094,
    607095, 607097, 607100, 607101, 607102, 607111, 607113, 607119, 607123, 607126,
    607146, 607147, 607195, 607246, 607236, 607273, 607276, 607277, 607285, 607294,
    607296, 607303, 607304, 607314, 607316, 607317, 607305, 607308, 607345, 607336,
    607338, 607380, 607382, 607440, 607584, 607595, 607605, 607606, 607607, 607608,
    607612, 607619, 607620, 607626, 607627, 607628, 607630, 607631, 607632, 607633,
    607635, 607636, 607637, 607639, 607656, 607657, 607667, 607695, 607723, 607724,
    607793, 607813, 607844, 607846, 607854, 607855, 607864, 607870, 607875, 607877,
    607878, 607901, 607905, 607906, 607911, 607923, 607924, 607926, 607931, 607933,
    607949, 607950, 607952, 607953, 607959, 607960, 607961, 607962, 607964, 607965,
    607966, 607968, 607969, 607970, 608056, 608094, 608112, 608118, 608119, 608122,
    608123, 608127, 608128, 608130, 608131, 608132, 608134, 608135, 608136, 608137,
    608146, 608147, 608150, 608162, 608163, 608172, 608173, 608176, 608177, 608180,
    608210, 608211, 608224, 608225, 608227, 608238, 608397, 608412, 608526, 608528,
    608530, 608531, 608532, 608534, 608535, 608536, 608548, 608875, 608876, 608923,
    608924, 608949, 608950, 609610, 610020, 610023, 610025, 610026, 610028, 610029,
    610063, 610268, 610384, 610442, 610584, 610611, 610619, 610625, 610671, 610672,
    610677, 610702, 610899, 610924, 610932, 610933, 610935, 610936, 610964, 610965,
    611050, 611128, 611162, 611202, 611213, 611214, 611239, 611238, 611248, 611259,
    611284, 611286, 611355, 611403, 611404, 611406, 611407, 611434, 611439, 611565,
    611584, 611585, 611590, 611591, 611592, 611603, 611605, 611606, 611607, 611608,
    611609, 611610, 611611, 611612, 611613, 611614, 611615, 611616, 611617, 611618,
    611619, 611620, 611622, 611624, 611625, 611626, 611627, 611628, 611630, 611631,
    611632, 611633, 611634, 611635, 611636, 611637, 611588, 611642, 611668, 611689,
    611694, 611695, 611703, 611705, 612078, 612083, 612085, 612087, 612089, 612091,
    612093, 612095, 612101, 612107, 612109, 612111, 612113, 612115, 612117, 612119,
    612121, 612047, 612048, 612049, 612050, 612051, 612052, 612053, 612054, 612055,
    612056, 612057, 612058, 612059, 612060, 612061, 612062, 612063, 612064, 612065,
    612066, 612067, 612068, 612069, 612070, 612071, 612072, 612073, 612074, 612025,
    612026, 612027, 612028, 612033, 612034, 612037, 612040, 612041, 612042, 612044,
    612045, 612046, 612134, 612136, 612138, 612140, 612176, 612177, 612305, 612306,
    612307, 612308, 612309, 612310, 612390, 612407
]


In [14]:
df = load_data(DATA_FILE)

results = []
failed = []

for sku in PHARMA_SKUS:
    try:
        pred = forecast_next_month_for_sku(sku, raw_df=df)

        if pred is not None:
            results.append(pred)
        else:
            failed.append({"ItemCode": sku, "Status": "No sequence / insufficient history"})

    except Exception as e:
        failed.append({"ItemCode": sku, "Status": str(e)})

forecast_df = pd.DataFrame(results)
failed_df = pd.DataFrame(failed)

print("Success:", len(forecast_df))
print("Failed:", len(failed_df))

forecast_df.head()

Success: 562
Failed: 124


,ItemCode,Forecast_Year,Forecast_Month,Expected_Bonus,Residual_Baseline,Predicted_Residual,GRU_Residual_Prediction,Recurring_Bonus_SKU,ABC_Class
0,600308,2026,2,0,24194.333333,-3637.835693,20556.497640,0,0
1,600310,2026,2,0,903.000000,-4.466538,898.533462,0,2
2,600311,2026,2,0,10648.666667,-1066.405518,9582.261149,0,0
3,600315,2026,2,0,1041.666667,-67.333008,974.333659,0,2
4,600319,2026,2,0,24617.666667,-363.406891,24254.259776,0,0


In [15]:
forecast_df[["ItemCode", "Forecast_Year", "Forecast_Month"]].drop_duplicates()

,ItemCode,Forecast_Year,Forecast_Month
0,600308,2026,2
1,600310,2026,2
2,600311,2026,2
3,600315,2026,2
4,600319,2026,2
...,...,...,...
557,611355,2026,2
558,611403,2026,2
559,611404,2026,2
560,611406,2026,2


In [16]:
forecast_df.to_csv("pharma_forecast_feb_2026.csv", index=False)

In [17]:
forecast_df.to_excel('pharma_forecast_feb_2026.xlsx', index=False)